# RQ2: native staged ApET versus matched pure random

This is the **next priority**, before broad input-only comparisons.
Six frozen questions / five images: full576, native96→32, native288→96,
and three pure-random seeds at each identical schedule. Compression is at input
and **after block16**. There are **54 primary predictions** plus separate controls.

Use **Internet and T4 ×2**, the existing **rq1-ih-inputs-v1** private Dataset,
and the locked original environment. Push code first; no new Dataset upload.
Native arms execute the original ApET input and decoder methods. Random performs
no FPS/E/basis replacement/merging at either boundary. Original forward, positions
and cache updates remain in charge of generation.

Set `RUN_STAGED_CALIBRATION = True` below, then run cells in order. Default False
prevents accidental background inference after import. Download ZIP + matching log.
This is an execution calibration on selected cases, not a benchmark ranking or
full paper reproduction. See `research/RQ2_STAGED_CALIBRATION.md`.


In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

RUN_STAGED_CALIBRATION = False  # Set True: six questions; native and matched random at96→32/288→96.
REPO_URL = "https://github.com/doguineyes/CSE590-ApET.git"
REPO_REF = "apet-analysis"
INPUT_ZIP = None  # Existing rq1-ih-inputs-v1 Dataset/ZIP; not the older rq1-inputs-v1.
CHECKPOINT_ARGS = []  # Optional existing original-format model and vision paths.

if not RUN_STAGED_CALIBRATION:
    raise RuntimeError("Set RUN_STAGED_CALIBRATION = True, then run cells manually.")
print("Preparing staged calibration code; model inference has not started.", flush=True)
repo = Path("/kaggle/working/CSE590-ApET")
if not repo.exists():
    subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True, timeout=120)
subprocess.run(["git", "fetch", "origin"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "checkout", REPO_REF], cwd=repo, check=True, timeout=120)
if REPO_REF == "apet-analysis":
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True, timeout=120)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo, check=True, timeout=120)
if not (repo / "scripts/run_rq2_staged_calibration.py").is_file():
    raise RuntimeError("Push the new staged calibration files first, then update this checkout.")
original_python = repo / "environments/original/.venv/bin/python"
rq2_staged_runs = []
print("Repository:", repo, flush=True)


## Prepare the existing input Dataset and locked runtime

This reuses the original environment/cache in a live session. Independent notebook sessions do not share virtual environments or `/tmp` checkpoints.


In [ ]:
sys.path.insert(0, str(repo / "scripts"))
from rq1_input_ih import locate_inputs
inputs = locate_inputs(
    [Path("/kaggle/input"), Path("/kaggle/working")],
    Path("/kaggle/working/rq1-ih-inputs-v1"), explicit=INPUT_ZIP,
)
print("Accepted input directory:", inputs, flush=True)
setup_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
setup_log = Path(f"/kaggle/working/rq2-staged-setup-{setup_stamp}.log")
runtime_report = Path(f"/kaggle/working/rq2-staged-runtime-{setup_stamp}.json")

def stream_command(command, stream, *, cwd=None):
    print("Running:", " ".join(map(str, command)), flush=True)
    stream.write("COMMAND " + repr(command) + "\n")
    stream.flush()
    process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8", bufsize=1)
    lines = []
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            stream.write(line)
            stream.flush()
            lines.append(line)
        status = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    stream.write(f"EXIT {status}\n")
    stream.flush()
    if status:
        raise RuntimeError(f"Command failed (exit {status}); preserve {stream.name} before changing anything.")
    return "".join(lines)

with setup_log.open("x", encoding="utf-8") as stream:
    stream_command([
        "bash", "bootstrap/setup_original_linux.sh", "--require-cuda", "--expected-gpus", "2",
        "--output", str(runtime_report),
    ], stream, cwd=repo)
print("Locked original runtime ready. Run the prerequisite/helper cell next.", flush=True)
print("Setup log:", setup_log)
print("Runtime report:", runtime_report)


## Run actual native prerequisites

Tiny18-block tests exercise both sides of block16, native versus direct generation,
random FPS/E/merge guards, exact frozen replays, full576 no-op and cached positions.
All tests must pass with **zero skips**. No model checkpoint downloads for these tests.


In [ ]:
def verify_rq2_staged_prerequisites(*, refresh_runtime=False):
    globals()["rq2_staged_prerequisites_passed"] = False
    with setup_log.open("a", encoding="utf-8") as stream:
        if refresh_runtime:
            stream_command([str(original_python), "-X", "utf8", "-I", "scripts/check_original_env.py",
                "--require-cuda", "--expected-gpus", "2", "--output", str(runtime_report)], stream, cwd=repo)
        output = stream_command([str(original_python), "-X", "utf8", "-m", "unittest",
            "research.tests.test_rq2_native", "research.tests.test_rq2_staged_protocol",
            "research.tests.test_rq2_staged_runner", "research.tests.test_rq2_staged_native", "-v"], stream, cwd=repo)
        count = re.search(r"Ran (\d+) tests?", output)
        if not count or int(count.group(1)) <= 10 or re.search(r"skipped\s*=", output):
            raise RuntimeError("Require all actual staged native tests and zero skips; preserve setup log.")
    command = [str(original_python), "-X", "utf8", "-I", "-c",
        "import json,sys; from pathlib import Path; sys.path.insert(0,str(Path.cwd()/'scripts')); "
        "from run_rq2_staged_calibration import prerequisite_signature,TESTS; "
        "print(json.dumps({'source_sha256':prerequisite_signature(),'test_modules':list(TESTS)}))"]
    signature = json.loads(subprocess.run(command, cwd=repo, capture_output=True, text=True,
                                          check=True, timeout=120).stdout)
    import hashlib
    proof = {"version": "rq2-staged-calibration-v1", "status": "passed", "tests_run": int(count.group(1)),
        "skipped": 0, **signature,
        "runtime_report_sha256": hashlib.sha256(runtime_report.read_bytes()).hexdigest(),
        "setup_log_sha256": hashlib.sha256(setup_log.read_bytes()).hexdigest()}
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    globals()["staged_prerequisite_report"] = Path(f"/kaggle/working/rq2-staged-prerequisites-{stamp}.json")
    staged_prerequisite_report.write_text(json.dumps(proof, indent=2) + "\n")
    globals()["rq2_staged_prerequisites_passed"] = True
    print("Staged prerequisites passed:", proof["tests_run"], "tests, zero skips.", flush=True)


def run_rq2_staged_calibration():
    if not globals().get("rq2_staged_prerequisites_passed", False):
        raise RuntimeError("Run verify_rq2_staged_prerequisites() successfully before loading pretrained weights.")
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    folder = Path(f"/kaggle/working/rq2-staged-calibration-v1-{stamp}")
    archive, log = folder.with_suffix(".zip"), folder.with_suffix(".log")
    globals().setdefault("rq2_staged_runs", []).append(folder)
    command = [str(original_python), "-X", "utf8", "-I", "scripts/run_rq2_staged_calibration.py",
        "--inputs", str(inputs), "--output", str(folder), "--calibration",
        "--require-cuda", "--expected-gpus", "2", "--download-budget-gib", "20",
        "--setup-log", str(setup_log), "--runtime-report", str(runtime_report),
        "--prerequisite-report", str(staged_prerequisite_report), *CHECKPOINT_ARGS]
    print("Run directory:", folder, flush=True)
    print("One model load; six questions ×9 primary conditions. Cached/no-op/fresh controls are additional.", flush=True)
    fallback_archive = False
    try:
        with log.open("x", encoding="utf-8") as stream:
            stream_command(command, stream, cwd=repo)
    finally:
        if not archive.exists():
            fallback_archive = True
            fallback = folder / "notebook-fallback"
            fallback.mkdir(parents=True, exist_ok=True)
            (fallback / "README.txt").write_text("Incomplete fallback: normal runner ZIP missing; no successful calibration established.\n")
            for path in [setup_log, runtime_report, staged_prerequisite_report, log]:
                if path.is_file():
                    shutil.copy2(path, fallback / path.name)
            with zipfile.ZipFile(archive, "x", zipfile.ZIP_DEFLATED) as bundle:
                for path in sorted(folder.rglob("*")):
                    if path.is_file():
                        bundle.write(path, path.relative_to(folder))
        print("Download:", archive, "and", log, flush=True)
        if (folder / "report.json").is_file():
            report = json.loads((folder / "report.json").read_text())
            for key in ("status", "calibration_passed", "progress", "wall_seconds", "summaries", "error"):
                if key in report:
                    print(key + ":", report[key], flush=True)
    if fallback_archive:
        raise RuntimeError("Runner ZIP missing; preserve fallback ZIP/log for diagnosis.")
    report = json.loads((folder / "report.json").read_text())
    if report.get("version") != "rq2-staged-calibration-v1" or report.get("status") != "passed" or not report.get("calibration_passed"):
        raise RuntimeError("Staged calibration did not pass; download ZIP/log for review.")
    print("Execution calibration passed. Review ZIP/log before broader staged evaluation.", flush=True)
    return folder


# Re-executing this cell invalidates the previous test pass.
globals()["rq2_staged_prerequisites_passed"] = False


In [ ]:
verify_rq2_staged_prerequisites()


## Run the small staged calibration

54 primary predictions, plus6 full576 identity controls,18 fixed replays,4 forced
four-token cache probes and4 fresh raw-native controls. Conditions run sequentially.
Original images and model canvases, native E/basis/merge graphs, random memberships,
cache traces, selected tensors, source/runtime evidence and raw answers are saved.
No compressed-accuracy threshold determines a pass.


In [ ]:
staged_calibration_dir = run_rq2_staged_calibration()


## Download ZIP and log before ending the session

Keep the timestamped **ZIP + matching log**. Setup/runtime/test proof are included
in the ZIP and linked separately. Preserve a failed/partial archive too. Review
calibration before expanding to TextVQA100/GQA development; the input-only100 runner
is an optional ablation and is currently deferred.


In [ ]:
from IPython.display import FileLink, display
paths = [setup_log, runtime_report, staged_prerequisite_report]
for folder in rq2_staged_runs:
    paths.extend([folder.with_suffix(".zip"), folder.with_suffix(".log")])
for path in paths:
    if path.exists():
        display(FileLink(str(path)))
